# ESPCN upscaler, reproducible run (v3)

This notebook retrains the corrected ESPCN model from `02_corrected_model.ipynb`
with the **same architecture, data and hyper-parameters**. It changes only what
stopped the earlier runs from being reproducible or comparable:

| Problem found in the re-analysis | Fix here |
|---|---|
| DataLoader workers drew crops from an unseeded NumPy state, so two runs trained on different patches | every crop and flip is a pure function of `(SEED, epoch, index)` |
| Non-deterministic CUDA kernels and TF32 | `torch.use_deterministic_algorithms`, cuDNN deterministic, TF32 off |
| The 10 test images also chose when to stop and whether to export | per-epoch checks use 10 **validation** images (0801–0810); the test images are touched once, at the end |
| v1 and v2 computed bicubic PSNR on different pixels | one `evaluate()` for every model, including the v1 INT8 model used on the laptop |
| `model_meta_v2.json` said 800 training images; the code used 120 | all metadata is computed from the objects that were used |
| Library versions were not recorded | versions, GPU, data hashes and model hashes are saved with the results |

**What "the same result" means.** On the same Colab GPU type with the same
library versions, two runs of this notebook should produce *identical* weights
(the weight hash is printed and compared). On a different GPU or PyTorch
version, results should agree within the tolerances in
`reproduce/expected/colab_expected.json` (for example ±0.15 dB PSNR). The last
cell checks this automatically.

Runtime → Change runtime type → **T4 GPU**. About 20 minutes, most of it the
DIV2K download.

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/hamzakazih1/npu-upscaling-placement"
REPO_BRANCH = "main"          # the branch holding reproduce/ (change while the PR is open)

SEED = 42
SCALE = 2
N_TRAIN_IMAGES = 120          # as in 02_corrected_model.ipynb
PATCHES_PER_IMAGE = 40
PATCH = 96
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3
NUM_WORKERS = 2
IN_H, IN_W = 270, 480         # the deployed (NPU) input shape

## 1. Determinism, before anything touches CUDA

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # required for deterministic cuBLAS
os.environ["PYTHONHASHSEED"] = str(SEED)

import hashlib, io, json, platform, random, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, __version__ as PIL_VERSION

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
# warn_only: an op without a deterministic kernel warns instead of crashing.
# None of the ops in this model should warn; if one does, the run is flagged.
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE,
      "| gpu", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none")
if DEVICE != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. "
          "CPU results will not match GPU results bit for bit.")

## 2. Pinned tools and the repository

In [ ]:
# Pinned, because INT8 calibration and export depend on these versions.
!pip install -q "onnx==1.17.0" "onnxruntime==1.20.1" onnxscript

import onnx, onnxruntime as ort
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__)

REPO = Path("/content/repo")
if not REPO.exists():
    !git clone -q --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO}
sys.path.insert(0, str(REPO / "reproduce" / "analysis"))
import stats_utils as su
print("repo at", subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

## 3. Data

Fixed, disjoint splits. Their file hashes are recorded so a different
download is detected.

| Split | Images | Used for |
|---|---|---|
| train | DIV2K train 0001–0120 | training patches |
| calibration | DIV2K train 0001–0064 | INT8 calibration (as in v2) |
| validation | DIV2K valid 0801–0810 | per-epoch check only |
| test | DIV2K valid 0891–0900 | the dissertation's 10 images, evaluated once |
| extended test | DIV2K valid 0811–0890 | 80 more images, for the quantisation-cost model |

In [ ]:
DATA = Path("/content/div2k"); DATA.mkdir(parents=True, exist_ok=True)
SOURCES = {
    "train": ("http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip", "DIV2K_train_HR"),
    "valid": ("http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", "DIV2K_valid_HR"),
}
for name, (url, folder) in SOURCES.items():
    if not (DATA / folder).exists():
        print(f"downloading {name}…")
        zip_path = DATA / f"{name}.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA)
        zip_path.unlink()

all_train = sorted((DATA / "DIV2K_train_HR").glob("*.png"))
all_valid = sorted((DATA / "DIV2K_valid_HR").glob("*.png"))
assert len(all_train) == 800 and len(all_valid) == 100, (len(all_train), len(all_valid))

train_images = all_train[:N_TRAIN_IMAGES]
calib_images = all_train[:64]
val_images = all_valid[:10]       # 0801-0810
test_images = all_valid[-10:]     # 0891-0900
ext_test_images = all_valid[10:90]  # 0811-0890
assert not set(val_images) & set(test_images) and not set(ext_test_images) & set(test_images)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def manifest(paths):
    h = hashlib.sha256()
    for p in paths:
        h.update(p.name.encode()); h.update(sha256_file(p).encode())
    return h.hexdigest()

DATA_MANIFEST = {
    "train": manifest(train_images), "calibration": manifest(calib_images),
    "validation": manifest(val_images), "test": manifest(test_images),
    "extended_test": manifest(ext_test_images),
}
for k, v in DATA_MANIFEST.items():
    print(f"{k:<14} {v[:16]}…")
print(f"train {train_images[0].name}–{train_images[-1].name} | "
      f"val {val_images[0].name}–{val_images[-1].name} | test {test_images[0].name}–{test_images[-1].name}")

## 4. Deterministic patch sampling

The v2 dataset called `np.random` inside DataLoader workers, whose NumPy state
was not seeded, so each run trained on different patches. Here every random
choice for sample `i` in epoch `e` comes from `default_rng((SEED, e, i))`: the
same patch, flip and rotation whichever worker loads it, in any order.

In [ ]:
from torch.utils.data import Dataset, DataLoader

def to_tensor(im):
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

class PatchDataset(Dataset):
    """Same sampling rules as v2 (variance rejection, flips, transpose), made deterministic."""

    def __init__(self, paths, patch=PATCH, scale=SCALE, samples=PATCHES_PER_IMAGE,
                 min_variance=0.02, max_tries=10):
        self.paths, self.patch, self.scale = paths, patch, scale
        self.samples, self.min_variance, self.max_tries = samples, min_variance, max_tries
        self.epoch = 0
        self.cache = {}

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.paths) * self.samples

    def _array(self, index):
        if index not in self.cache:
            self.cache[index] = np.asarray(Image.open(self.paths[index]).convert("RGB"), dtype=np.uint8)
        return self.cache[index]

    def __getitem__(self, i):
        rng = np.random.default_rng((SEED, self.epoch, i))
        arr = self._array(i // self.samples)
        h, w, _ = arr.shape
        crop = None
        for _ in range(self.max_tries):
            y = int(rng.integers(0, h - self.patch)); x = int(rng.integers(0, w - self.patch))
            y -= y % self.scale; x -= x % self.scale
            candidate = arr[y:y + self.patch, x:x + self.patch]
            if candidate.mean(axis=2).std() / 255.0 >= self.min_variance:
                crop = candidate
                break
        if crop is None:
            crop = candidate
        hr_img = Image.fromarray(crop)
        lr_img = hr_img.resize((self.patch // self.scale, self.patch // self.scale), Image.BICUBIC)
        hr, lr = to_tensor(hr_img), to_tensor(lr_img)
        flips = rng.random(3)
        if flips[0] < 0.5: hr, lr = torch.flip(hr, [2]), torch.flip(lr, [2])
        if flips[1] < 0.5: hr, lr = torch.flip(hr, [1]), torch.flip(lr, [1])
        if flips[2] < 0.5: hr, lr = hr.transpose(1, 2), lr.transpose(1, 2)
        return lr.contiguous(), hr.contiguous()

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

train_ds = PatchDataset(train_images)
loader_gen = torch.Generator().manual_seed(SEED)
# persistent_workers=False so each epoch's workers receive the updated epoch number.
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                      drop_last=True, persistent_workers=False, worker_init_fn=seed_worker,
                      generator=loader_gen)

# Proof of determinism: the same index gives the same patch twice.
a, b = train_ds[123], train_ds[123]
assert torch.equal(a[0], b[0]) and torch.equal(a[1], b[1])
print(f"{len(train_ds):,} patches | {len(train_dl)} batches per epoch | sampling is deterministic")

## 5. Model (unchanged from v2)

In [ ]:
class ESPCN(nn.Module):
    """ESPCN with a frozen anchor residual. Operators: Conv, Relu, Add, Clip, DepthToSpace."""

    def __init__(self, scale=SCALE, channels=48):
        super().__init__()
        out_ch = 3 * scale * scale
        self.conv1 = nn.Conv2d(3, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv3 = nn.Conv2d(channels, channels // 2, 3, padding=1)
        self.conv4 = nn.Conv2d(channels // 2, out_ch, 3, padding=1)
        self.anchor = nn.Conv2d(3, out_ch, 1, bias=False)
        with torch.no_grad():
            weight = torch.zeros(out_ch, 3, 1, 1)
            for i in range(out_ch):
                weight[i, i // (scale * scale), 0, 0] = 1.0
            self.anchor.weight.copy_(weight)
        self.anchor.weight.requires_grad_(False)
        self.shuffle = nn.PixelShuffle(scale)

    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = F.relu(self.conv2(y))
        y = F.relu(self.conv3(y))
        y = self.conv4(y) + self.anchor(x)
        return self.shuffle(torch.clamp(y, 0.0, 1.0))

torch.manual_seed(SEED)
model = ESPCN().to(DEVICE)
TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)
TOTAL_PARAMS = sum(p.numel() for p in model.parameters())

def weights_hash(m):
    h = hashlib.sha256()
    for name, t in sorted(m.state_dict().items()):
        h.update(name.encode()); h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

INIT_HASH = weights_hash(model)
print(f"trainable {TRAINABLE:,} | total {TOTAL_PARAMS:,} | initial weights {INIT_HASH[:16]}…")

## 6. One evaluation protocol for every model

Identical to the v2 notebook's final evaluation, so results stay comparable
with the dissertation: downscale the full HR image by 2 with PIL bicubic, take
the top-left 270×480 of the LR image and the matching 540×960 of the HR image,
and compare the model and PyTorch bicubic upsampling against it, as RGB PSNR.

In [ ]:
def psnr(a, b):
    mse = F.mse_loss(a.clamp(0, 1), b.clamp(0, 1))
    return (10 * torch.log10(1.0 / mse)).item()

@torch.no_grad()
def evaluate(net, paths, label=""):
    if hasattr(net, "eval"):
        net.eval()
    rows = []
    for path in paths:
        hr_img = Image.open(path).convert("RGB")
        lr_img = hr_img.resize((hr_img.width // SCALE, hr_img.height // SCALE), Image.BICUBIC)
        lr_t = to_tensor(lr_img)[None][:, :, :IN_H, :IN_W].to(DEVICE)
        hr_t = to_tensor(hr_img)[None][:, :, :IN_H * SCALE, :IN_W * SCALE].to(DEVICE)
        pred = net(lr_t).clamp(0, 1)
        bic = F.interpolate(lr_t, scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)
        rows.append({"name": path.name, "model_psnr": psnr(pred, hr_t), "bicubic_psnr": psnr(bic, hr_t)})
    df = pd.DataFrame(rows)
    df["gain"] = df.model_psnr - df.bicubic_psnr
    if label:
        print(f"{label:<14} model {df.model_psnr.mean():.3f} dB | bicubic {df.bicubic_psnr.mean():.3f} dB | "
              f"gain {df.gain.mean():+.3f} dB | beats bicubic {(df.gain > 0).sum()}/{len(df)}")
    return df

class OnnxModel:
    """An ONNX file behind the same call interface, run on CPU."""
    def __init__(self, path):
        so = ort.SessionOptions(); so.log_severity_level = 3
        so.intra_op_num_threads = 1      # fixed threading -> identical float reductions
        self.session = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.name = self.session.get_inputs()[0].name
    def __call__(self, x):
        out = self.session.run(None, {self.name: x.cpu().numpy()})[0]
        return torch.from_numpy(out).to(x.device)

## 7. Training (validation images only)

In [ ]:
def train(model, loader, epochs=EPOCHS, lr=LR):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(loader))
    history = []
    for epoch in range(1, epochs + 1):
        train_ds.set_epoch(epoch)
        model.train()
        running, batches, start = 0.0, 0, time.time()
        for lr_b, hr_b in loader:
            lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
            loss = F.l1_loss(model(lr_b), hr_b)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            running += loss.item(); batches += 1
        v = evaluate(model, val_images)
        history.append({"epoch": epoch, "loss": running / batches, "val_model_psnr": v.model_psnr.mean(),
                        "val_bicubic_psnr": v.bicubic_psnr.mean(), "val_gain": v.gain.mean()})
        print(f"epoch {epoch}/{epochs}  L1 {running/batches:.5f}  val gain {v.gain.mean():+.3f} dB  "
              f"{time.time()-start:.0f}s")
    return pd.DataFrame(history)

import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    history = train(model, train_dl)
DET_WARNINGS = sorted({str(w.message)[:200] for w in caught if "determinis" in str(w.message).lower()})
if DET_WARNINGS:
    print("!! non-deterministic ops were used; exact repeatability is not guaranteed:", DET_WARNINGS)
FINAL_HASH = weights_hash(model)
print(f"\ntrained weights {FINAL_HASH[:16]}…  (identical runs give an identical hash)")
assert history.val_gain.iloc[-1] > 0, "loses to bicubic on validation images: do not export"

## 8. Test evaluation (first and only time the test images are used)

In [ ]:
eval_fp32 = evaluate(model, test_images, "fp32 test")
ext_fp32 = evaluate(model, ext_test_images, "fp32 ext-test")

## 9. Export, operator audit, INT8 quantisation (QDQ, as for the HTP backend)

In [ ]:
from onnxruntime.quantization import (quantize_static, CalibrationDataReader, QuantType,
                                      QuantFormat, CalibrationMethod)
from onnxruntime.quantization.shape_inference import quant_pre_process

EXPORT = Path("/content/export_v3"); EXPORT.mkdir(exist_ok=True)
ONNX_PATH = EXPORT / "espcn_x2_v3.onnx"
PREPROC = EXPORT / "espcn_x2_v3_preproc.onnx"
INT8_PATH = EXPORT / "espcn_x2_v3_int8.onnx"

model.eval().cpu()
dummy = torch.rand(1, 3, IN_H, IN_W, generator=torch.Generator().manual_seed(SEED))
torch.onnx.export(model, dummy, ONNX_PATH, input_names=["input"], output_names=["output"],
                  opset_version=13, do_constant_folding=True, export_params=True, dynamo=False)
model.to(DEVICE)
graph = onnx.load(ONNX_PATH); onnx.checker.check_model(graph)
OPS = sorted({n.op_type for n in graph.graph.node})
with torch.no_grad():
    diff = np.abs(model.cpu()(dummy).numpy() - OnnxModel(ONNX_PATH)(dummy).numpy()).max()
model.to(DEVICE)
print("operators:", OPS, f"| max |torch - onnx| {diff:.2e}")
assert diff < 1e-4

class ImageCalibrationReader(CalibrationDataReader):
    """Unchanged from v2: 64 real LR crops from the first 64 training images, fixed RNG."""
    def __init__(self, paths, input_name, n=64, h=IN_H, w=IN_W):
        rng = np.random.default_rng(SEED)
        self.samples = []
        for i in range(n):
            img = Image.open(paths[i % len(paths)]).convert("RGB")
            lr = img.resize((img.width // SCALE, img.height // SCALE), Image.BICUBIC)
            arr = np.asarray(lr, dtype=np.float32) / 255.0
            if arr.shape[0] < h or arr.shape[1] < w:
                lr = lr.resize((max(w, lr.width), max(h, lr.height)), Image.BICUBIC)
                arr = np.asarray(lr, dtype=np.float32) / 255.0
            y = rng.integers(0, arr.shape[0] - h + 1); x = rng.integers(0, arr.shape[1] - w + 1)
            self.samples.append({input_name: arr[y:y+h, x:x+w].transpose(2, 0, 1)[None].astype(np.float32)})
        self.iterator = iter(self.samples)
    def get_next(self): return next(self.iterator, None)
    def rewind(self): self.iterator = iter(self.samples)

quant_pre_process(str(ONNX_PATH), str(PREPROC), skip_symbolic_shape=True)
reader = ImageCalibrationReader(calib_images, OnnxModel(PREPROC).name)
quantize_static(model_input=str(PREPROC), model_output=str(INT8_PATH), calibration_data_reader=reader,
                quant_format=QuantFormat.QDQ, activation_type=QuantType.QUInt8, weight_type=QuantType.QUInt8,
                per_channel=False, calibrate_method=CalibrationMethod.MinMax,
                extra_options={"ActivationSymmetric": False, "WeightSymmetric": True})
INT8_OPS = sorted({n.op_type for n in onnx.load(INT8_PATH).graph.node})
print("int8 operators:", INT8_OPS)

## 10. INT8 evaluation, plus the v1 model under the same protocol

In [ ]:
print("ONNX on CPU, a few minutes…")
eval_int8 = evaluate(OnnxModel(INT8_PATH), test_images, "int8 test")
ext_int8 = evaluate(OnnxModel(INT8_PATH), ext_test_images, "int8 ext-test")
# The model the laptop latency and placement results were measured with.
eval_v1 = evaluate(OnnxModel(REPO / "models" / "v1" / "espcn_x2_int8.onnx"), test_images, "v1 int8 test")

## 11. Statistics

The paired estimates from the re-analysis, and the quantisation-cost model
refitted on the 80 extended-test images. With n = 80, this settles whether the
slope seen on 10 images is real.

In [ ]:
def cost_frame(f, i):
    q = f.merge(i, on="name", suffixes=("_fp32", "_int8"))
    q["cost"] = q.model_psnr_int8 - q.model_psnr_fp32
    return q

q10, q80 = cost_frame(eval_fp32, eval_int8), cost_frame(ext_fp32, ext_int8)
fit = su.ols(q80.cost, [np.ones(len(q80)), q80.bicubic_psnr_fp32], ["intercept", "bicubic_psnr"])
quad = su.ols(q80.cost, [np.ones(len(q80)), q80.bicubic_psnr_fp32, q80.bicubic_psnr_fp32 ** 2],
              ["intercept", "bicubic_psnr", "bicubic_psnr_sq"])
print(su.coefs_table(fit))
print("diagnostics:", {k: round(v, 4) for k, v in su.residual_checks(fit).items()})
print("quadratic term F, p:", su.nested_f(fit, quad))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history.epoch, history.val_gain, "o-"); ax[0].axhline(0, c="k", lw=.6)
ax[0].set(xlabel="epoch", ylabel="validation gain over bicubic (dB)", title="Training")
ax[1].scatter(q80.bicubic_psnr_fp32, q80.cost, s=12, label="extended test (80)")
ax[1].scatter(q10.bicubic_psnr_fp32, q10.cost, s=18, label="test (10)")
xx = np.linspace(q80.bicubic_psnr_fp32.min(), q80.bicubic_psnr_fp32.max(), 10)
b = fit["coefs"]
ax[1].plot(xx, b["intercept"]["estimate"] + b["bicubic_psnr"]["estimate"] * xx, "r--")
ax[1].set(xlabel="bicubic PSNR (dB)", ylabel="INT8 - fp32 PSNR (dB)", title="Quantisation cost"); ax[1].legend()
plt.tight_layout(); plt.savefig(EXPORT / "colab_figures.png", dpi=130); plt.show()

## 12. Save everything, with provenance

In [ ]:
def gpu_info():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "none"

summary = {
    "environment": {
        "python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(), "gpu": gpu_info(), "numpy": np.__version__,
        "pillow": PIL_VERSION, "onnx": onnx.__version__, "onnxruntime": ort.__version__,
        "platform": platform.platform(),
    },
    "config": {"seed": SEED, "scale": SCALE, "train_images": len(train_images), "patches_per_image": PATCHES_PER_IMAGE,
               "patch": PATCH, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "num_workers": NUM_WORKERS,
               "input_shape": [1, 3, IN_H, IN_W], "opset": 13},
    "data_manifest": DATA_MANIFEST,
    "model": {"trainable_parameters": int(TRAINABLE), "total_parameters": int(TOTAL_PARAMS),
              "operators": OPS, "int8_operators": INT8_OPS,
              "init_weights_sha256": INIT_HASH, "final_weights_sha256": FINAL_HASH,
              "fp32_onnx_sha256": sha256_file(ONNX_PATH), "int8_onnx_sha256": sha256_file(INT8_PATH)},
    "training_history": history.to_dict(orient="list"),
    "test": {
        "psnr_fp32": eval_fp32.model_psnr.mean(), "psnr_int8": eval_int8.model_psnr.mean(),
        "psnr_bicubic": eval_fp32.bicubic_psnr.mean(),
        "gain_fp32": su.paired_mean_ci(eval_fp32.gain), "gain_int8": su.paired_mean_ci(eval_int8.gain),
        "int8_cost": su.paired_mean_ci(q10.cost),
        "int8_beats_bicubic": int((eval_int8.gain > 0).sum()),
        "v1_int8_psnr": eval_v1.model_psnr.mean(), "v1_int8_gain": su.paired_mean_ci(eval_v1.gain),
    },
    "extended_test": {
        "gain_fp32": su.paired_mean_ci(ext_fp32.gain), "gain_int8": su.paired_mean_ci(ext_int8.gain),
        "int8_cost": su.paired_mean_ci(q80.cost),
        "cost_model": {"coefs": fit["coefs"], "r2": fit["r2"], "adj_r2": fit["adj_r2"],
                       "diagnostics": su.residual_checks(fit), "quadratic_f_p": su.nested_f(fit, quad)},
    },
    "deterministic_warnings": DET_WARNINGS,
}
def tidy(o):
    if isinstance(o, dict): return {k: tidy(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [tidy(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
summary = tidy(summary)
(EXPORT / "colab_summary.json").write_text(json.dumps(summary, indent=2))
for name, df in [("eval_fp32_v3", eval_fp32), ("eval_int8_v3", eval_int8), ("eval_v1_int8_same_protocol", eval_v1),
                 ("ext_eval_fp32_v3", ext_fp32), ("ext_eval_int8_v3", ext_int8), ("training_history_v3", history)]:
    df.to_csv(EXPORT / f"{name}.csv", index=False)
torch.save({"state_dict": model.state_dict(), "summary": summary}, EXPORT / "checkpoint_v3.pt")
print(json.dumps(summary["test"], indent=2))

## 13. Check against the expected results

Compares this run with `reproduce/expected/colab_expected.json` (tolerance
bands taken from the dissertation's v2 run) and, if present, with
`reproduce/expected/colab_reference.json` (an earlier run of this notebook,
which should match exactly on the same GPU and versions).

To make this run the reference for future runs, download
`colab_summary.json` and commit it as `reproduce/expected/colab_reference.json`.

In [ ]:
!python {REPO}/reproduce/analysis/compare.py colab {EXPORT}/colab_summary.json

In [ ]:
import shutil
shutil.make_archive("/content/export_v3", "zip", EXPORT)
try:
    from google.colab import files
    files.download("/content/export_v3.zip")
except ImportError:
    print("zip at /content/export_v3.zip")